# 🕸️ Fundamentos de LangGraph: State, Nodes y Edges

> **Nivel:** Iniciación 🟢
> **Tiempo de lectura:** ~8 minutos

---

## 🎯 ¿Qué vamos a aprender?

LangGraph es una forma nueva de construir aplicaciones de IA, mucho más flexible que las "chains" (cadenas) tradicionales de LangChain. En vez de un flujo en línea recta, LangGraph funciona como un **mapa de decisiones**: la información puede ir, volver, ramificarse y acumularse.

En esta lectura veremos sus **3 piezas fundamentales**:

| Componente | En una frase |
|---|---|
| 🧠 **State** | La memoria compartida que todos pueden leer y escribir |
| ⚙️ **Nodes** | Las tareas individuales que hacen el trabajo |
| 🔀 **Edges** | Los caminos que deciden qué pasa después |

---

## 🆚 Chain vs. StateGraph: la diferencia clave

Para entender por qué LangGraph es tan potente, comparemos las dos arquitecturas:

**Chain lineal (lo tradicional):**
```
Input → Paso A → Output A → Paso B → Output B → Paso C → Resultado final
```
Aquí cada paso solo conoce lo que el paso anterior le entregó. Si el Paso C necesita algo del Paso A, hay que pasarlo manualmente por todos los intermedios. 😩

**StateGraph (LangGraph):**
```
              🧠 Estado Global Compartido 🧠
                 ↕         ↕         ↕
Input →      Nodo A  ←→  Nodo B  ←→  Nodo C      → Output
```
Aquí **todos los nodos comparten una misma "pizarra"** (el estado). Cualquier nodo puede leer lo que otro escribió, sin importar cuándo se ejecutó. 🎉

> 💡 **Analogía simple:** Piensa en el State como un documento de Google Docs compartido. No importa quién lo edite (Nodo A, B o C), todos ven la versión más actualizada.

Esto habilita cosas que una chain simple no puede hacer fácilmente: **memoria persistente, loops, y decisiones condicionales**.

---

## 1️⃣ State (Estado): la memoria compartida

### ¿Qué es?

El **State** es la información que vive durante toda la ejecución del grafo. Es como una **base de datos temporal** que todos los nodos pueden consultar y actualizar.

### 🔑 Sus 4 características principales

| Característica | Qué significa |
|---|---|
| **Global** | Cualquier nodo puede leer cualquier dato del estado, sin que se lo "pasen" explícitamente |
| **Mutable** | Se va actualizando conforme los nodos trabajan |
| **Tipado** | Se define con una estructura clara (ej. `TypedDict`), como si fuera un formulario con campos fijos |
| **Flexible** | Puede guardar desde un simple texto hasta listas, objetos o resultados de búsquedas |

### 📝 Ejemplo de un estado típico

```python
pregunta: str
respuesta: str
intentos: int
documentos_relevantes: List[str]
```

Este "contrato" le dice a **todos los nodos** qué información existe y qué forma tiene, para que sepan con qué están trabajando.

### ✅ ¿Por qué es tan útil?

- **Memoria de conversación** → mantiene el contexto entre pasos
- **Acumulación de resultados** → varios nodos pueden ir sumando información
- **Decisiones informadas** → un nodo puede decidir en base a todo lo que pasó antes
- **Flujo flexible** → no hay que definir rígidamente qué entra y sale de cada paso

---

## 2️⃣ Nodes (Nodos): las unidades de trabajo

### ¿Qué es un nodo?

Un **nodo** es simplemente **una función** que:

1. 📥 Recibe el estado completo
2. ⚡ Hace su tarea (buscar info, generar texto, validar algo, etc.)
3. 📤 Devuelve **solo la parte del estado que modificó**

> 💡 **Punto clave:** un nodo NO devuelve todo el estado de nuevo, solo lo nuevo o lo que cambió. LangGraph se encarga de mezclarlo automáticamente con el resto.

### 🔄 ¿Qué hace LangGraph por detrás?

```
1. Toma el estado actual
2. Se lo pasa al nodo
3. Recibe la actualización parcial
4. La fusiona con el estado global
5. Pasa el estado ya actualizado al siguiente nodo
```

Así, cada nodo se concentra **solo en su tarea**, sin preocuparse de cargar con todo lo demás.

### 🧩 Tipos de nodos que puedes encontrar

| Tipo | Qué hace | Ejemplo |
|---|---|---|
| 🔧 **Procesamiento** | Transforma datos existentes | Analizar sentimiento de un texto |
| ✍️ **Generación** | Crea contenido nuevo (usa LLM) | Redactar una respuesta |
| 🔍 **Recuperación** | Trae información externa | Buscar en una base de datos o API |
| 🚦 **Decisión** | Evalúa condiciones | Marcar si el resultado es válido o no |
| ✅ **Validación** | Verifica calidad/corrección | Calcular un puntaje de confianza |

### 📐 Buenas prácticas al diseñar nodos

- **Responsabilidad única** → un nodo, una tarea clara
- **Atomicidad** → debe poder ejecutarse de forma completa e independiente
- **Actualizaciones mínimas** → solo devuelve lo que cambió
- **Determinismo** → si es posible, mismo input = mismo output (ayuda a debuggear)

---

## 3️⃣ Edges (Aristas): el control de flujo

### ¿Qué son?

Los **edges** son las flechas que conectan los nodos y deciden **qué se ejecuta después**. A diferencia de una simple tubería, pueden ser **condicionales**, es decir, tomar decisiones sobre la marcha.

### 🔀 Dos tipos de edges

| Tipo | Descripción |
|---|---|
| **➡️ Fijos (normales)** | Siempre van del Nodo A al Nodo B, sin excepciones |
| **🔀 Condicionales** | Una función revisa el estado y decide a cuál nodo ir después |

### 🚩 Nodos especiales: START y END

- **START** → el punto de entrada. Solo recibe el estado inicial, no hace nada más.
- **END** → el punto de salida. Marca que el flujo terminó y devuelve el resultado final.

### 🚀 Lo que los edges condicionales hacen posible

- 🔁 **Reintentos (retry loops):** si algo sale mal, se vuelve a un nodo anterior
- 🌳 **Ramificación por calidad:** distintos caminos según qué tan bueno sea el resultado
- 📈 **Escalamiento gradual:** empezar simple, y si no alcanza, usar un método más potente
- ⚡ **Paralelización condicional:** ejecutar varias ramas al mismo tiempo si el estado lo permite

---

## 🧭 Consideraciones al diseñar tu grafo

| Aspecto | Por qué importa |
|---|---|
| **Planificación del estado** | Define bien qué información necesitas y cómo evolucionará |
| **Gestión de complejidad** | Un grafo demasiado grande es difícil de debuggear y mantener |
| **Funciones reducer** | Definen cómo combinar actualizaciones cuando dos nodos escriben en la misma clave *(lo veremos en clases futuras)* |

---

## 📌 Resumen: lo esencial para recordar

> - 🧠 **State** = memoria compartida y persistente entre nodos
> - ⚙️ **Nodes** = funciones que solo devuelven lo que cambian
> - 🔀 **Edges** = flechas que controlan el flujo, incluso de forma condicional
> - 🕸️ **StateGraph** = la combinación de los tres, que habilita workflows no lineales, adaptativos y con memoria

Con estos tres bloques, LangGraph permite construir aplicaciones que:

- ✅ Adaptan su comportamiento dinámicamente
- ✅ Mantienen contexto complejo a largo plazo
- ✅ Manejan flujos iterativos y de refinamiento
- ✅ Toman decisiones complejas basadas en múltiples factores

---



# Construcción del primer grafo

```mermaid
graph LR
    subgraph Estado["🧠 State (TypedDict)"]
        direction TB
        s1["texto_original: str"]
        s2["texto_mayus: str"]
        s3["longitud: int"]
    end

    START((START)) -->|"texto_original = 'Hola mundo'"| Mayus[Mayus]
    Mayus -->|"escribe texto_mayus"| Contar[Contar]
    Contar -->|"escribe longitud"| END((END))

    Mayus -.lee/escribe.-> Estado
    Contar -.lee/escribe.-> Estado

    style START fill:#2ecc71,color:#fff
    style END fill:#e74c3c,color:#fff
    style Mayus fill:#3498db,color:#fff
    style Contar fill:#3498db,color:#fff
    style Estado fill:#f4f4f4,color:#000
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
# 1. Definir el esquema del estado
class State(TypedDict):
    texto_original: str
    texto_mayus: str
    longitud: int

In [ ]:
# 2. Crear el grafo de estado
graph = StateGraph(State)

In [ ]:
# 3. Definir las funciones de los nodos
def poner_mayusculas(state):
    texto = state['texto_original']
    return {"texto_mayus": texto.upper()}

def contar_caracteres(state):
    texto = state['texto_mayus']
    return {"longitud": len(texto)}

In [ ]:
# 4. Añadir los nodos al grafo
graph.add_node("Mayus", poner_mayusculas)
graph.add_node("Contar", contar_caracteres)

In [ ]:
# 5. Conectar los nodos en secuencia
graph.add_edge(START, "Mayus")
graph.add_edge("Mayus", "Contar")
graph.add_edge("Contar", END)

In [ ]:
# 6. Compilar el grafo
compiled_graph = graph.compile()

In [ ]:
# 7. Invocar el grafo con un estado inicial
estado_inicial = {"texto_original": "Hola mundo"}
resultado = compiled_graph.invoke(estado_inicial)
print(resultado)

# Annotated types

## 🏷️ Annotated Types en LangGraph

### 🤔 El problema que resuelve

Por defecto, cuando un nodo devuelve un valor para un campo del estado, LangGraph **sobreescribe** el valor anterior con el nuevo. Esto funciona bien cuando cada nodo calcula un valor desde cero (como `texto_mayus` o `longitud`).

Pero ¿qué pasa si en vez de reemplazar quieres **acumular**? Por ejemplo, una lista de logs donde cada nodo agrega un mensaje sin borrar los anteriores. Ahí es donde entra `Annotated`.

### 📦 ¿Qué es `Annotated`?

`Annotated` es una utilidad de Python (de la librería estándar `typing`) que permite decirle a un tipo: *"eres una lista, pero además ten en cuenta esta información extra"*.

```python
Annotated[TIPO, METADATO]
```

En LangGraph, ese metadato extra es una **función reducer**: le indica al grafo *cómo combinar* el valor viejo con el valor nuevo, en vez de simplemente reemplazarlo.

> 💡 **Analogía:** `Annotated` es como una nota pegada al campo del estado que dice *"cuando alguien traiga un valor nuevo para este campo, no lo reemplaces, combínalo con esta regla"*.

### 🔍 Sin Annotated vs. con Annotated

**Sin Annotated (comportamiento por defecto → sobreescribe):**

```python
class State(TypedDict):
    logs: list[str]

def nodo_1(state):
    return {"logs": ["Nodo 1 ejecutado"]}

def nodo_2(state):
    return {"logs": ["Nodo 2 ejecutado"]}
```

Resultado final: `logs = ["Nodo 2 ejecutado"]` ❌ *(se pierde el mensaje del Nodo 1)*

**Con Annotated (usando `add` como reducer → acumula):**

```python
from typing import Annotated
from operator import add

class State(TypedDict):
    logs: Annotated[list[str], add]
```

Resultado final: `logs = ["Nodo 1 ejecutado", "Nodo 2 ejecutado"]` ✅ *(se conservan ambos)*

`add` es el operador `+` de Python (de `operator`). Igual que `["a"] + ["b"]` da `["a", "b"]`, LangGraph usa esa lógica para **concatenar** listas en cada actualización, en vez de sobreescribirlas.


In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END

In [ ]:
# 1. Definir el esquema del estado
class State(TypedDict):
    texto_original: str
    texto_mayus: str
    longitud: int
    logs: Annotated[list[str], add]  # 👈 este campo se ACUMULA, no se reemplaza

In [ ]:
# 2. Crear el grafo de estado
graph = StateGraph(State)

In [ ]:
# 3. Definir las funciones de los nodos
def poner_mayusculas(state):
    texto = state['texto_original']
    return {
        "texto_mayus": texto.upper(),
        "logs": [f"✅ Se convirtió '{texto}' a mayúsculas"]
    }

def contar_caracteres(state):
    texto = state['texto_mayus']
    return {
        "longitud": len(texto),
        "logs": [f"✅ Se contaron {len(texto)} caracteres"]
    }

In [ ]:
# 4. Añadir los nodos al grafo
graph.add_node("Mayus", poner_mayusculas)
graph.add_node("Contar", contar_caracteres)

In [ ]:
# 5. Conectar los nodos en secuencia
graph.add_edge(START, "Mayus")
graph.add_edge("Mayus", "Contar")
graph.add_edge("Contar", END)

In [ ]:
# 6. Compilar el grafo
compiled_graph = graph.compile()

In [ ]:
# 7. Invocar el grafo con un estado inicial
estado_inicial = {"texto_original": "Hola mundo", "logs": []}
resultado = compiled_graph.invoke(estado_inicial)
print(resultado)

# Routing condicional en LangGraph

En un grafo normal, un nodo siempre le entrega el control al mismo siguiente nodo. Pero muchas veces necesitamos que el flujo **decida dinámicamente**, según el contenido del estado, hacia dónde continuar. A eso se le llama **routing condicional**, y es el mecanismo que le permite a un grafo comportarse como un `if/else`, pero de forma explícita y visible en la estructura del grafo, no escondido dentro de un nodo gigante.

### Qué resuelve el routing condicional

Sin este mecanismo, tendríamos que meter toda la lógica de decisión dentro de un solo nodo enorme, que calcula, evalúa condiciones y decide qué hacer, todo mezclado. El routing condicional separa esas responsabilidades:

- **El nodo hace un trabajo concreto** (por ejemplo, sumar dos números) y entrega un estado actualizado.
- **Una función de routing** simplemente *lee* ese estado y decide, según una condición, hacia qué nodo debe continuar el flujo. Esta función no modifica el estado, solo retorna una etiqueta que indica la rama a seguir.
- **El grafo usa esa etiqueta** para saltar al nodo correspondiente, que a su vez hace su propio trabajo.

Esto le da al grafo tres ventajas importantes:

1. **Trazabilidad**: se puede ver claramente por cuál rama pasó la ejecución (por ejemplo, "Suma → Impar"), en vez de una caja negra.
2. **Escalabilidad**: si en el futuro se necesitan más ramas (no solo dos), basta con agregar más opciones a la decisión, sin reescribir la lógica interna de los nodos.
3. **Separación de responsabilidades**: los nodos se enfocan en transformar el estado; la función de routing se enfoca únicamente en decidir el camino.

### Qué hace el ejemplo

El grafo recibe dos números (`numero1` y `numero2`). El primer nodo los suma y guarda el resultado en el estado. Luego, en vez de ir directo a un solo nodo final, el grafo evalúa una condición sobre esa suma: si es par, sigue por una rama; si es impar, sigue por otra. Cada rama arma un mensaje final indicando si el resultado fue par o impar, y ambas terminan convergiendo en el mismo punto final del grafo.

Por ejemplo, si `numero1 = 5` y `numero2 = 6`, la suma da `11`. Como `11` es impar, el grafo elige la rama correspondiente y el resultado final indica que "el número sumado 11 es impar".

### Diagrama del flujo

```mermaid
flowchart TD
    START([START]) --> Suma[Suma<br/>numero1 + numero2]
    Suma --> Decision{suma % 2 == 0?}
    Decision -- "Par" --> Par[Par<br/>arma mensaje: es par]
    Decision -- "Impar" --> Impar[Impar<br/>arma mensaje: es impar]
    Par --> END([END])
    Impar --> END
```

### Idea clave para recordar

El routing condicional no es un nodo más del grafo: es una **regla de decisión** que conecta un nodo origen con varios posibles nodos destino, basada en el estado actual. El nodo hace el trabajo, la función de routing decide el camino, y el grafo se encarga de saltar al nodo correcto.


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

d:\Repositorios\Github\curso_ia\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# Definir el estado
class State(TypedDict):
    numero1: int
    numero2: int
    suma: int   
    resultado_par_impar: str

graph = StateGraph(State)

In [ ]:
# Definir los nodos del workflow
def caso_suma(state):
    suma = state['numero1'] + state['numero2']
    return {'suma': suma}

def caso_par(state):
    return {f'resultado_par_impar': f'El numero sumado {state["suma"]} es par'}

def caso_impar(state):
    return {'resultado_par_impar': f'El numero sumado {state["suma"]} es impar'}

In [4]:
# Añadir los nodos al grafo
graph.add_node("Suma", caso_suma)
graph.add_node("Par", caso_par)
graph.add_node("Impar", caso_impar)

In [5]:
# Definir la funcion de routing para decidir la rama de ejecucion
def decidir_rama(state):
    if state["suma"] % 2 == 0:
        return "Par"
    else:
        return "Impar"

In [6]:
# Conectar los nodos en secuencia
graph.add_edge(START, "Suma")
graph.add_conditional_edges(
    "Suma", 
    decidir_rama,
    {
        "Par": "Par",
        "Impar": "Impar"
    }
)
graph.add_edge("Par", END)
graph.add_edge("Impar", END)

In [ ]:
compiled = graph.compile()

In [9]:
# Probar el grafo con ejemplos
print(compiled.invoke({"numero1": 5, "numero2": 6})["resultado_par_impar"])

El numero sumado 11 es impar
